# SSL Experiment — nb03: CTLS-SSL Training

**Goal:** Train CTLS-SSL (phase-gated SimCLR + CircuitConsistencyLoss on mined neighbors) and verify training dynamics before the few-shot evaluation in nb04.

**Phase structure:**
- Phase 1 (epochs 0–49): SimCLR only — bootstraps output embedding, populates bank
- Phase 2 (epochs 50–199): SimCLR + CircuitConsistencyLoss on cross-instance neighbors

**Key diagnostics:**
1. Loss curves across both phases — circuit loss should activate and decrease in Phase 2
2. Lambda and tau schedules — verify expected shapes
3. Bank quality check — are mined neighbor pairs visually coherent?
4. Linear probe and KNN-20 (same protocol as nb01/nb02)

## 0. Setup

In [ ]:
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = 'https://github.com/jacobposchl/model-interpretability'
    REPO_DIR = '/content/ctls'

    if not os.path.exists(REPO_DIR):
        os.system(f"git clone {REPO_URL} {REPO_DIR}")
        os.system(f"pip install -r {REPO_DIR}/requirements.txt -q")

    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_BASE = '/content/drive/MyDrive/ctls'
    os.makedirs(f"{DRIVE_BASE}/data/raw", exist_ok=True)
    os.makedirs(f"{DRIVE_BASE}/experiments", exist_ok=True)

    for link, target in [
        (f"{REPO_DIR}/data/raw",    f"{DRIVE_BASE}/data/raw"),
        (f"{REPO_DIR}/experiments", f"{DRIVE_BASE}/experiments"),
    ]:
        if os.path.islink(link): os.unlink(link)
        os.symlink(target, link)

    ROOT = REPO_DIR
else:
    ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
print(f"Working directory: {os.getcwd()} ({'Colab' if IN_COLAB else 'local'})")

In [ ]:
import yaml

with open('configs/ssl/ctls_ssl.yaml') as f:
    config = yaml.safe_load(f)

print(yaml.dump(config, default_flow_style=False))

## 1. Train CTLS-SSL

Expected training time: ~50–70 min on GPU (200 epochs). Phase 2 is slower per step
due to neighbor retrieval and an extra backbone forward pass.

**Skip if `experiments/ssl/ctls_ssl/best.pt` already exists.**

In [ ]:
from training.ssl_trainer import SSLTrainer

trainer = SSLTrainer(config)
trainer.train()

## 2. Schedule Diagnostics

Verify that lambda and tau schedules have the correct shape.
- Lambda should be 0 for epochs 0–49, ramp from 0→1 over epochs 50–69, then hold at 1.
- Tau should be 1.0 for Phase 1, then anneal from 1.0→0.1 over Phase 2.

In [ ]:
from training.schedulers import LambdaScheduler, TauScheduler
import matplotlib.pyplot as plt

tcfg = config['training']
warmup_phase = tcfg['warmup_phase_epochs']
total_epochs = tcfg['epochs']

lambda_sched = LambdaScheduler(
    init_val=tcfg['lambda_consistency']['init'],
    final_val=tcfg['lambda_consistency']['final'],
    warmup_epochs=tcfg['lambda_consistency']['warmup_epochs'],
)
tau_sched = TauScheduler(
    init_val=tcfg['temperature']['init'],
    final_val=tcfg['temperature']['final'],
    anneal_epochs=tcfg['temperature']['anneal_epochs'],
)

epochs = list(range(total_epochs))
# Phase 1: lambda=0, tau=init. Phase 2: use relative epoch.
lambdas = [0.0 if e < warmup_phase else lambda_sched.get(e - warmup_phase) for e in epochs]
taus = [tcfg['temperature']['init'] if e < warmup_phase else tau_sched.get(e - warmup_phase) for e in epochs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, lambdas, lw=2, color='steelblue')
ax1.axvline(warmup_phase, color='gray', linestyle='--', lw=1, label=f'Phase boundary (ep {warmup_phase})')
ax1.set(xlabel='Epoch', ylabel='λ', title='Circuit loss weight (λ)')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs, taus, lw=2, color='darkorange')
ax2.axvline(warmup_phase, color='gray', linestyle='--', lw=1)
ax2.set(xlabel='Epoch', ylabel='τ', title='Soft mask temperature (τ)')
ax2.grid(True, alpha=0.3)

fig.tight_layout()
os.makedirs('experiments/ssl/ctls_ssl', exist_ok=True)
fig.savefig('experiments/ssl/ctls_ssl/schedules.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Bank Quality Check

Visualize a sample of mined nearest-neighbor pairs from Phase 2.
If the bank is working correctly, neighbor pairs should be visually/semantically similar
even though no class labels were used — cars paired with cars, animals with animals.

In [ ]:
import torch
import torch.nn.functional as F
from models.soft_mask import SoftMask
from models.backbone import CTLSBackbone
from models.meta_encoder import MetaEncoder
from models.momentum_encoder import EmbeddingBank
from data.ssl import get_multiview_loaders, get_ssl_augmentation
from torchvision.datasets import CIFAR10
import numpy as np
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

mcfg = config['model']; ecfg = mcfg['meta_encoder']
soft_mask = SoftMask(init_temperature=mcfg.get('soft_mask', {}).get('init_temperature', 1.0))
backbone = CTLSBackbone(arch=mcfg['arch'], num_classes=mcfg['num_classes'],
                         soft_mask=soft_mask, pretrained=False).to(DEVICE)
meta_encoder = MetaEncoder(
    layer_dims=backbone.layer_dims,
    hidden_dim=ecfg.get('hidden_dim', 256),
    embedding_dim=ecfg.get('embedding_dim', 64),
    encoder_type=ecfg.get('encoder_type', 'mlp'),
).to(DEVICE)

ckpt = torch.load('experiments/ssl/ctls_ssl/best.pt', map_location=DEVICE)
backbone.load_state_dict(ckpt['backbone_state'])
meta_encoder.load_state_dict(ckpt['meta_encoder_state'])
backbone.eval(); meta_encoder.eval()

# Restore bank from checkpoint
bank_cfg = config['training']['embedding_bank']
bank = EmbeddingBank(bank_size=bank_cfg['bank_size'],
                     embedding_dim=ecfg['embedding_dim'], device=DEVICE).to(DEVICE)
if 'bank_state' in ckpt:
    bank.load_state_dict(ckpt['bank_state'])
    print(f"Bank ready: {bank.is_ready()}")
else:
    print("Bank state not in checkpoint — rebuild by running a forward pass.")

print(f"Loaded CTLS-SSL checkpoint (epoch {ckpt['epoch']})")

In [ ]:
# Visualize neighbor pairs
from data.cifar import get_val_transform

raw_cifar = CIFAR10(root=config['data']['data_dir'], train=True, transform=None, download=False)
val_transform = get_val_transform()

CIFAR10_CLASSES = ['airplane','automobile','bird','cat','deer',
                   'dog','frog','horse','ship','truck']

# Pick 8 random anchor images and find their bank nearest neighbors
anchor_indices = torch.randint(0, len(raw_cifar), (8,))
anchor_imgs = torch.stack([val_transform(raw_cifar[i][0]) for i in anchor_indices]).to(DEVICE)

with torch.no_grad():
    _, traj = backbone(anchor_imgs)
    z_anchors = meta_encoder(traj)   # [8, 64]

if bank.is_ready():
    _, nbr_indices = bank.get_neighbors(
        z_anchors, k=1, exclude_self_indices=anchor_indices.to(DEVICE)
    )
    nbr_indices_flat = nbr_indices[:, 0].cpu()

    fig, axes = plt.subplots(2, 8, figsize=(18, 5))
    for col in range(8):
        a_idx = anchor_indices[col].item()
        n_idx = nbr_indices_flat[col].item()

        a_img, a_label = raw_cifar[a_idx]
        n_img, n_label = raw_cifar[n_idx]

        axes[0, col].imshow(np.array(a_img))
        axes[0, col].set_title(f'Anchor\n{CIFAR10_CLASSES[a_label]}', fontsize=8)
        axes[0, col].axis('off')

        axes[1, col].imshow(np.array(n_img))
        match = '✓' if a_label == n_label else '✗'
        axes[1, col].set_title(f'Neighbor {match}\n{CIFAR10_CLASSES[n_label]}', fontsize=8)
        axes[1, col].axis('off')

    fig.suptitle('CTLS-SSL: Bank nearest-neighbor pairs (no labels used)', fontsize=12)
    fig.tight_layout()
    fig.savefig('experiments/ssl/ctls_ssl/neighbor_pairs.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Bank not fully populated — run more training epochs or rebuild.")

## 4. Linear Probe and KNN-20

In [ ]:
from data.cifar import get_standard_loaders
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import TensorDataset, DataLoader as DL
from sklearn.neighbors import KNeighborsClassifier

dcfg = config['data']
train_loader_labeled, val_loader = get_standard_loaders(
    data_dir=dcfg['data_dir'], batch_size=256, num_workers=4, augment=True, download=True,
)

all_z_train, all_y_train = [], []
all_z_val, all_y_val = [], []
with torch.no_grad():
    for x, y in train_loader_labeled:
        _, traj = backbone(x.to(DEVICE))
        all_z_train.append(meta_encoder(traj).cpu()); all_y_train.append(y)
    for x, y in val_loader:
        _, traj = backbone(x.to(DEVICE))
        all_z_val.append(meta_encoder(traj).cpu()); all_y_val.append(y)

z_tr = torch.cat(all_z_train); y_tr = torch.cat(all_y_train)
z_val = torch.cat(all_z_val); y_val = torch.cat(all_y_val)

EMBEDDING_DIM = ecfg.get('embedding_dim', 64)
probe = nn.Linear(EMBEDDING_DIM, 10).to(DEVICE)
opt = AdamW(probe.parameters(), lr=1e-3, weight_decay=1e-4)
ds = DL(TensorDataset(z_tr, y_tr), batch_size=256, shuffle=True)

for ep in range(100):
    probe.train()
    for z, y in ds:
        loss = F.cross_entropy(probe(z.to(DEVICE)), y.to(DEVICE))
        opt.zero_grad(); loss.backward(); opt.step()

probe.eval()
with torch.no_grad():
    preds = probe(z_val.to(DEVICE)).argmax(dim=-1).cpu()
lp_acc = (preds == y_val).float().mean().item()

knn = KNeighborsClassifier(n_neighbors=20, metric='cosine')
knn.fit(z_tr.numpy(), y_tr.numpy())
knn_acc = (knn.predict(z_val.numpy()) == y_val.numpy()).mean()

print(f"Linear probe accuracy (CTLS-SSL): {lp_acc:.4f}")
print(f"KNN-20 accuracy      (CTLS-SSL): {knn_acc:.4f}")

## 5. Circuit Silhouette

In [ ]:
from evaluation.circuit_viz import CircuitVisualizer
import torch.nn.functional as F

viz = CircuitVisualizer(backbone, meta_encoder, val_loader, DEVICE)
fig = viz.plot_umap(
    title='CTLS-SSL — circuit embedding space',
    compare_output=False, max_samples=5000,
)
fig.savefig('experiments/ssl/ctls_ssl/umap_circuit.png', dpi=150, bbox_inches='tight')
plt.show()

scores = viz.cluster_separation_score(max_samples=5000)
print(f"Circuit silhouette (CTLS-SSL): {scores['silhouette_circuit']:.4f}")

## Summary

| Metric | SimCLR | DINO | CTLS-SSL |
|--------|--------|------|---------|
| Linear probe accuracy | ___ | ___ | ___ |
| KNN-20 accuracy | ___ | ___ | ___ |
| Circuit silhouette | ___ | ___ | ___ |

**Next:** Run `nb04_fewshot_eval.ipynb` — the main comparative evaluation.